# Depuracion de features sobre `panel_todo`

Parte de las 62 series que arma `leer_todo.py` y llega a un conjunto de features defendible.

El orden importa: primero se van las que son **el target disfrazado** (fuga), luego los
**duplicados exactos**, y solo al final se desempata entre series correlacionadas.
Hacerlo al reves esconde la fuga dentro de un grupo.

> Para modelado final usa `panel_d.parquet`: su target es `precio_ponderado`
> (ponderado por demanda), no el promedio simple de las 24 horas que usa `panel_todo`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

RAIZ = Path(r"C:\Users\andre\OneDrive\cosas de la universidad\Uniandes\VIII\No one noticed\pronostico-precio-bolsa-colombia")
TARGET = "precio_bolsa_horario_prom"
INICIO = "2015-01-01"
CORTE_SELECCION = "2019-12-31"

df = pd.read_parquet(RAIZ / "data" / "panel_todo.parquet")
print(df.shape)
df.tail(3)

## 1. Recorte temporal y tipos de faltante

El panel arranca en 1950 por el ONI y en 1991 por la TRM, pero las series de XM
empiezan en 2015. Antes de mirar nulos hay que recortar, si no toda serie de XM
parece 70% vacia solo por el rango.

Despues del recorte, **no todos los NaN significan lo mismo**. Tirar por umbral de
nulos sin mirar el mecanismo bota series buenas y deja entrar basura:

| mecanismo | como se reconoce | tratamiento |
|---|---|---|
| **evento que no ocurrio** | 0% de ceros y `min` >> 0: XM no publica fila si no hubo | `fillna(0)` |
| **baja frecuencia** | ~96.7% NaN = un dato cada 30 dias (mensual) | `ffill` |
| **sin datos en la ventana** | 100% NaN en el periodo | excluir |

El caso de `vertimientos` es el ejemplo claro: 48% de NaN, cero ceros y minimo
43.87. No es que falte el dato — es que esos dias **no hubo vertimiento**.
Imputarlo con la media (como haria una receta de dataset transversal) inventaria
vertimientos que nunca ocurrieron.

In [ ]:
df = df[df.index >= INICIO]

# --- diagnostico del mecanismo de cada faltante ---------------------------
ventana = df[df.index <= CORTE_SELECCION]
diag = pd.DataFrame({
    "nan": ventana.isna().mean(),
    "ceros": (ventana == 0).mean(),
    "min": ventana.min(),
})
print(diag[diag["nan"] > 0.02].sort_values("nan", ascending=False).to_string())

# --- 1) evento que no ocurrio -> 0 ----------------------------------------
# XM no publica fila los dias sin racionamiento / sin intercambio / sin vertimiento.
EVENTO_CERO = [
    "dem_no_atendida_noprog", "demanda_no_atendida",
    "exportaciones", "importaciones", "vertimientos",
]

# --- 2) publicacion mensual -> arrastrar el ultimo dato conocido -----------
# ffill solo mira hacia atras, no hay fuga.
BAJA_FRECUENCIA = [
    "oni", "cee", "cere", "mc",
    "demanda_upme_alto", "demanda_upme_bajo", "demanda_upme_medio",
    "precio_escasez", "precio_escasez_act", "precio_escasez_pon",
    "precio_marg_escasez", "precio_cargo_conf", "emisiones_co2_sistema",
]

for col in EVENTO_CERO:
    if col in df.columns:
        df[col] = df[col].fillna(0)
for col in BAJA_FRECUENCIA:
    if col in df.columns:
        df[col] = df[col].ffill()

# --- 3) sin datos en la ventana de seleccion -> fuera ----------------------
vacias = ventana.columns[ventana.isna().mean() > 0.995]
print(f"\nsin datos en {INICIO[:4]}-{CORTE_SELECCION[:4]}: {list(vacias)}")
df = df.drop(columns=vacias)

print(f"\n{df.shape[1]} series, NaN restante: {df.isna().mean().mean():.2%}")

## 2. Duplicados exactos

`leer_todo.py` lee `cache_api/` y `data/`, y varias series estan en las dos:
los CSV `*_Sistema.csv` son el mismo dato que el parquet correspondiente.
Se detectan solos con correlacion 1.0.

In [ ]:
c = df.corr()

duplicados = []
for i in range(len(c)):
    for j in range(i + 1, len(c)):
        if abs(c.iloc[i, j]) > 0.9999:
            duplicados.append((c.index[i], c.columns[j], c.iloc[i, j]))

for a, b, r in duplicados:
    print(f"{a:32s} == {b:32s} r={r:+.6f}")

## 3. Auditoria de fuga

El paso que de verdad decide si la tesis vale. Una exogena que correlaciona >0.9
con el precio **del dia siguiente** casi nunca es una buena predictora: suele ser
el mismo precio por otro nombre.

En el mercado colombiano el precio de bolsa **es** el costo marginal del despacho
ideal. Entonces `costo_marginal_prog`, `max_precio_oferta` y `precio_oferta_desp`
no son variables explicativas: son la definicion del target.

In [ ]:
y = df[TARGET]
X = df.drop(columns=[TARGET])

audit = pd.DataFrame({
    "r_hoy": X.corrwith(y),
    "r_manana": X.corrwith(y.shift(-1)),
}).sort_values("r_manana", key=abs, ascending=False)

audit["sospecha_fuga"] = audit["r_manana"].abs() > 0.70
audit.head(15)

## 4. Grupos correlacionados: niveles vs. diferencias

Agrupar sobre **niveles** infla los grupos: dos series no estacionarias con
tendencia comun dan r alto sin tener relacion real (la TRM y el precio del cargo
por confiabilidad dan r=0.98 solo por eso).

Se compara contra el agrupamiento sobre **primeras diferencias**, que es el que
hay que creer.

In [ ]:
def agrupar(datos, umbral=0.90):
    corr = datos.corr().fillna(0)
    dist = 1 - corr.abs()
    np.fill_diagonal(dist.values, 0)
    Z = linkage(squareform(dist.values, checks=False), "average")
    return pd.Series(fcluster(Z, 1 - umbral, "distance"), index=datos.columns)

g_niv = agrupar(X)
g_dif = agrupar(X.diff())

print(f"grupos en niveles:     {g_niv.nunique():3d}")
print(f"grupos en diferencias: {g_dif.nunique():3d}  <- el que vale")

for k in sorted(g_dif.unique()):
    miembros = g_dif[g_dif == k].index.tolist()
    if len(miembros) > 1:
        mejor = max(miembros, key=lambda m: abs(audit.loc[m, "r_manana"]))
        print(f"\ngrupo {k}:")
        for m in miembros:
            marca = "  <-- representante" if m == mejor else ""
            print(f"    {m:32s} r_manana={audit.loc[m, 'r_manana']:+.3f}{marca}")

## 5. Conjunto final

Las listas quedan explicitas a proposito: cada exclusion es una decision
defendible en la tesis, no el resultado opaco de un umbral.

In [ ]:
# El precio de bolsa ES el costo marginal del despacho: estas lo replican.
# El precio pasado si entra, pero como rezago autorregresivo, no como exogena.
FUGA = [
    "costo_marginal_prog",
    "max_precio_oferta",
    "precio_oferta_desp",
    "PrecBolsNaci_Sistema_csv",
]

# Misma serie leida de cache_api/ y de data/
DUPLICADAS = [
    "AporEner_Sistema_csv",
    "CapaUtilDiarEner_Sistema_csv",
    "VoluUtilDiarEner_Sistema_csv",
    "DemaCome_Sistema_csv",
    "Gene_Sistema_csv",
]

# Redundantes: se conserva el representante indicado al lado
REDUNDANTES = [
    "demanda_no_regulada", "demanda_por_or", "demanda_regulada",
    "gen_por_recurso",              # -> gen_programada
    "disp_comercial", "disp_real",  # -> disp_declarada (la ex-ante)
    "precio_cargo_conf",            # -> trm
    "capacidad_efectiva", "precio_cont_no_regu",  # -> precio_prom_contrato
    "precio_escasez", "precio_escasez_pon", "precio_marg_escasez",
                                    # -> precio_escasez_act
    "volumen_por_embalse", "volumen_util_pct",    # -> volumen_util_pct_emb
    "emisiones_co2_sistema",        # -> consumo_combustible
    "restricciones_sin_aliv",       # -> restricciones
    "aportes_pct_rio",              # -> aportes_pct_sistema
    "irradiacion_global",           # -> irradiacion_panel
]

X_final = X.drop(columns=FUGA + DUPLICADAS + REDUNDANTES, errors="ignore")

print(f"{X.shape[1]} -> {X_final.shape[1]} features")
print()
for col in X_final.columns:
    print("  ", col)

## 5b. Cobertura, medida despues de los rezagos

Un umbral de nulos sobre la serie cruda es enganoso. Al construir 6 rezagos +
`d7`, **cada dia faltante envenena 7 columnas derivadas en 7 filas distintas**:
una serie con 80% de cobertura dispersa puede dejar el 70% de las filas
incompletas despues del `dropna()` conjunto.

Por eso la cobertura hay que medirla sobre la **matriz de features ya rezagada**,
que es justo donde la mide `higiene()` en `01_seleccion_variables.py`
(`max_nan_frac=0.30`), no sobre el panel crudo.

Con la imputacion por mecanismo del paso 1 esto ya quedo casi resuelto: lo que
sobrevive aca son huecos reales de calendario.

In [ ]:
# cobertura que queda DESPUES de imputar por mecanismo
cobertura = X_final[X_final.index <= CORTE_SELECCION].notna().mean().sort_values()

print(f"cobertura en {INICIO[:4]}-{CORTE_SELECCION[:4]} (peores 8):\n")
print(cobertura.head(8).to_string(float_format=lambda v: f"{v:.1%}"))
print(f"\n{X_final.shape[1]} features entran a la etapa de rezagos")
print("el filtro por cobertura real se aplica despues de rezagar (paso 7)")

## 6. Rezagos

Todas las que quedan son **contemporaneas y ex-post**: el consumo de combustible
del dia D no se conoce al pronosticar D. Con lag 0 cualquiera de estas es fuga.

Regla: toda estadistica movil lleva `shift(1)` **antes** del `rolling`, nunca
centrada. Asi el valor en t solo usa informacion hasta t-1.

Las unicas que pueden ir con lag 0 son las ex-ante validadas en
`01_seleccion_variables.py` (`ex_ante_cols`), donde ya esta razonado serie por
serie cuando publica XM cada dato.

In [ ]:
LAGS = [1, 2, 3, 7, 14, 30]
VENTANAS = [7, 30]

log_y = np.log(y.clip(lower=1e-6))
log_y.name = "y"

feats = {}

# bloque autorregresivo: aqui es donde entra el precio pasado
for L in LAGS:
    feats[f"log_precio_lag{L}"] = log_y.shift(L)
for w in VENTANAS:
    feats[f"log_precio_ma{w}"] = log_y.shift(1).rolling(w, min_periods=w // 2).mean()
    feats[f"log_precio_sd{w}"] = log_y.shift(1).rolling(w, min_periods=w // 2).std()

# exogenas: lag >= 1 sin excepcion
for col in X_final.columns:
    s = X_final[col]
    for L in LAGS:
        feats[f"{col}_lag{L}"] = s.shift(L)
    feats[f"{col}_d7"] = s.shift(1) - s.shift(8)

XF = pd.DataFrame(feats, index=df.index)
print(f"{X_final.shape[1]} series -> {XF.shape[1]} features con rezagos")
XF.tail(3)

## 7. Corte temporal

Mismos periodos que el resto del pipeline, para que las metricas sean
comparables entre etapas:

| periodo | rango | uso |
|---|---|---|
| seleccion | 2015-2019 | tamizaje de variables |
| validacion | 2020-2022 | eleccion de bloques |
| prueba | 2023-2025 | solo medicion final |

In [ ]:
MAX_NAN_FEATURE = 0.30

mask = XF.index <= CORTE_SELECCION
X_sel, y_sel = XF[mask], log_y[mask]

# higiene sobre la matriz ya rezagada: primero se botan COLUMNAS con demasiado
# hueco, y solo despues se botan FILAS. Al reves, una sola columna mala se
# lleva por delante casi todo el periodo.
nan_col = X_sel.isna().mean()
malas = nan_col[nan_col > MAX_NAN_FEATURE].index
if len(malas):
    print(f"se van {len(malas)} features con >{MAX_NAN_FEATURE:.0%} NaN:")
    print(f"  {sorted({m.rsplit('_lag', 1)[0].rsplit('_d7', 1)[0] for m in malas})}")
X_sel = X_sel.drop(columns=malas)

antes = len(X_sel)
juntos = pd.concat([X_sel, y_sel], axis=1).dropna()
X_sel, y_sel = juntos.drop(columns="y"), juntos["y"]

print(f"\nfilas: {antes} -> {len(X_sel)} tras dropna conjunto")
print(f"seleccion: {X_sel.shape[0]} dias x {X_sel.shape[1]} features")
print(f"rango:     {X_sel.index.min().date()} -> {X_sel.index.max().date()}")

## 8. Informacion mutua

Captura relaciones no lineales que la correlacion de Pearson se pierde.
Se calcula **solo sobre el periodo de seleccion**: si viera 2020-2022, la
comparacion posterior de bloques quedaria sesgada.

In [ ]:
from sklearn.feature_selection import mutual_info_regression

mi = pd.Series(
    mutual_info_regression(X_sel, y_sel, random_state=42),
    index=X_sel.columns,
    name="mi",
).sort_values(ascending=False)

mi.head(25).to_frame()

## 9. Trampa: la MI premia a lo que sirve de reloj

En el ranking de arriba, justo debajo de los rezagos del precio, aparecen todas
las series **mensuales** (`oni`, `mc`, `cere`, `cee`, `demanda_upme_*`) con
MI ~1.05-1.10. **No es senal, es un artefacto.**

`mutual_info_regression` usa un estimador de k vecinos mas cercanos. Una serie
mensual rellenada con `ffill` tiene ~60 valores unicos repartidos en 1789 filas:
hay unas 30 copias identicas de cada valor, la distancia al vecino mas cercano es
0, y la MI estimada se dispara.

| feature | valores unicos | MI | corr. Pearson |
|---|---|---|---|
| `mc_lag1` | 60 | **1.069** | **-0.094** |
| `cere_lag1` | 60 | 1.093 | -0.378 |
| `consumo_combustible_lag1` | 1789 | **0.795** | **+0.894** |

Una variable con correlacion -0.09 no puede ser mejor predictora que una con
+0.89. Lo que mide la MI ahi es "que mes es", no el valor de la serie.

**El mismo problema aparece disfrazado de continua.** `precio_prom_contrato`
tiene 1789 valores unicos, asi que pasa el filtro de cardinalidad, y saca
MI=0.955 con apenas r=-0.107. La razon: su correlacion **con el tiempo mismo** es
**0.968** — es practicamente un indice temporal. Mientras tanto el log del precio
correlaciona -0.287 con el tiempo. La MI esta capturando "en que punto de la serie
estamos", no una relacion economica.

**Implicacion:** la MI cruda no sirve para rankear estas features entre si. Hay
que separarlas por cardinalidad (celda siguiente) y, sobre todo, desconfiar de
toda serie con tendencia fuerte: la solucion es evaluarlas ya diferenciadas, que
es justo lo que hace `estacionarizar()` en `01_seleccion_variables.py` antes de
correr Granger.

In [ ]:
ESCALON_MAX_UNICOS = 200  # menos valores unicos que esto = serie de escalon

diag_mi = pd.DataFrame({
    "mi": mi,
    "unicos": X_sel.nunique(),
    "r": X_sel.corrwith(y_sel),
})
diag_mi["tipo"] = np.where(diag_mi["unicos"] < ESCALON_MAX_UNICOS,
                           "escalon", "continua")

for tipo in ["continua", "escalon"]:
    sub = diag_mi[diag_mi["tipo"] == tipo].sort_values("mi", ascending=False)
    print(f"--- {tipo.upper()} ({len(sub)} features) ---")
    print(sub.head(10)[["mi", "unicos", "r"]].to_string(
        float_format=lambda v: f"{v:8.3f}"))
    print()

## 10. Lasso (metodo embedded)

`embedded()` vive en `01_seleccion_variables.py`, no en este notebook. Como el
script protege la ejecucion con `if __name__ == "__main__":`, importarlo aqui
no dispara el pipeline completo, solo trae la funcion.

In [ ]:
import importlib.util
import matplotlib.pyplot as plt

# "01_seleccion_variables" empieza por numero, no es un identificador valido
# para import normal -> se carga por ruta de archivo.
spec = importlib.util.spec_from_file_location(
    "seleccion_variables", RAIZ / "01_seleccion_variables.py")
seleccion_variables = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seleccion_variables)

emb = seleccion_variables.embedded(X_sel, y_sel)
emb.head()

In [ ]:
top = emb["lasso_abs"].dropna().sort_values(ascending=False).head(25)
n_vivas = int((emb["lasso_abs"].dropna() > 0).sum())

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(top.index[::-1], top.values[::-1], color="steelblue")
ax.set_xlabel("|coeficiente| (features estandarizadas)")
ax.set_title(f"Lasso: top 25 de {emb['lasso_abs'].notna().sum()} features "
             f"({n_vivas} sobreviven)")
plt.tight_layout()
plt.show()

## 11. Como decide un arbol del Random Forest

`embedded()` usa un bosque de **300 arboles**, no uno solo — la idea del RF es
que cada arbol individual sobreajusta, pero el promedio de 300 entrenados con
muestras/features distintas cancela ese ruido. Por eso no tiene sentido
graficar "el" arbol: hay que elegir uno.

`embedded()` no devuelve el modelo entrenado (solo las dos columnas de
importancia), asi que aqui se reentrena un RF con los mismos parametros y el
mismo split 80/20 cronologico, solo para poder extraer un arbol y mirarlo.

El arbol real tiene muchos mas niveles (`min_samples_leaf=5` es la unica cota:
sigue partiendo mientras cada hoja tenga 5+ observaciones). Se grafica **cortado
a 3 niveles** porque un arbol completo con 200+ features es ilegible — esto
muestra solo las primeras decisiones, las de mayor peso.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import plot_tree

# mismo filtro de cobertura y misma imputacion que adentro de embedded()
cobertura_rf = X_sel.notna().mean()
cols_ok = cobertura_rf[cobertura_rf >= 0.80].index.tolist()
Xd = X_sel[cols_ok].fillna(X_sel[cols_ok].median()).values
yd = y_sel.values

n_tr = int(len(Xd) * 0.8)
rf_ilustrativo = RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                        n_jobs=-1, random_state=42)
rf_ilustrativo.fit(Xd[:n_tr], yd[:n_tr])

arbol = rf_ilustrativo.estimators_[0]
print(f"arbol 1 de {rf_ilustrativo.n_estimators}: "
      f"profundidad real = {arbol.get_depth()}, {arbol.get_n_leaves()} hojas")

fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(arbol, feature_names=cols_ok, max_depth=3, fontsize=9,
          filled=True, rounded=True, ax=ax)
ax.set_title("Primer arbol del RF, primeros 3 niveles "
             f"(de {arbol.get_depth()} reales)")
plt.tight_layout()
plt.show()

## 12. La regresion de Lasso, con signo

`embedded()` guarda `|coeficiente|` porque eso es lo que sirve para el conteo
de votos (importa que aporte, no en que direccion). Aqui se reentrena igual
pero se guarda `lasso.coef_` completo, para ver signo y ajuste.

**Ojo con el R2 que sale abajo**: es un ajuste **in-sample** (mismos datos con
los que se entreno) y `log_precio_lag1` domina el coeficiente. Un R2 alto aca
es en gran parte autocorrelacion del propio precio, no evidencia de que las
exogenas aporten. La prueba real esta en `02_evaluacion_bloques.py`: error
walk-forward comparado por bloque con Diebold-Mariano.

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

cobertura_l = X_sel.notna().mean()
cols_l = cobertura_l[cobertura_l >= 0.80].index.tolist()
Xd_l = X_sel[cols_l].fillna(X_sel[cols_l].median())

sc = StandardScaler().fit(Xd_l)
tscv = TimeSeriesSplit(n_splits=5)
try:
    lasso = LassoCV(cv=tscv, random_state=42, alphas=50, max_iter=5000) \
        .fit(sc.transform(Xd_l), y_sel)
except TypeError:
    lasso = LassoCV(cv=tscv, random_state=42, n_alphas=50, max_iter=5000) \
        .fit(sc.transform(Xd_l), y_sel)

coef_con_signo = pd.Series(lasso.coef_, index=cols_l, name="coef")
vivos = coef_con_signo[coef_con_signo != 0].sort_values(key=abs, ascending=False)
r2 = lasso.score(sc.transform(Xd_l), y_sel)

print(f"alpha elegido por CV: {lasso.alpha_:.5f}")
print(f"intercepto:           {lasso.intercept_:+.4f}")
print(f"R2 in-sample:         {r2:.3f}  (ver nota arriba)")
print(f"\n{len(vivos)} de {len(cols_l)} features con coeficiente != 0:\n")
print(vivos.to_string(float_format=lambda v: f"{v:+.4f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(vivos))))
colores = ["seagreen" if v > 0 else "indianred" for v in vivos.values[::-1]]
ax.barh(vivos.index[::-1], vivos.values[::-1], color=colores)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("coeficiente (features estandarizadas; verde sube el precio, rojo lo baja)")
ax.set_title(f"Lasso: {len(vivos)} features vivas, alpha={lasso.alpha_:.4f}")
plt.tight_layout()
plt.show()

pred = lasso.predict(sc.transform(Xd_l))
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_sel.index, y_sel.values, label="log(precio) real", lw=1)
ax.plot(y_sel.index, pred, label="ajuste Lasso", lw=1, alpha=0.8)
ax.legend()
ax.set_title(f"Ajuste in-sample sobre el periodo de seleccion, R2={r2:.3f}")
plt.tight_layout()
plt.show()